### MAJORITY VOTING


In [19]:
# ============================================================
# 1. IMPORT LIBRARY
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

import matplotlib.pyplot as plt

print("Library berhasil di-import.")

Library berhasil di-import.


### Load Dataset

membuat pipeline voting ensemble dari hasil prediksi tiga model, lalu mengevaluasi performanya terhadap label asli dari data test.

In [20]:
# ============================================================
# 2. SETUP PATH
# ============================================================

BASE_DIR = Path("..").resolve()

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = BASE_DIR / "outputs"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

TEST_PREDICTIONS_PATH = PROCESSED_DIR / "test_predictions.csv"

print("Setup path berhasil.")
print("BASE_DIR             :", BASE_DIR)
print("PROCESSED_DIR        :", PROCESSED_DIR)
print("OUTPUTS_DIR          :", OUTPUTS_DIR)
print("TEST_PREDICTIONS_PATH:", TEST_PREDICTIONS_PATH)

Setup path berhasil.
BASE_DIR             : E:\project-kda-kelompok3
PROCESSED_DIR        : E:\project-kda-kelompok3\data\processed
OUTPUTS_DIR          : E:\project-kda-kelompok3\outputs
TEST_PREDICTIONS_PATH: E:\project-kda-kelompok3\data\processed\test_predictions.csv


In [21]:
# ============================================================
# 3. CEK FILE DATASET
# ============================================================

if TEST_PREDICTIONS_PATH.exists():
    print("test_predictions.csv ditemukan.")
else:
    print("test_predictions.csv tidak ditemukan:", TEST_PREDICTIONS_PATH)

test_predictions.csv ditemukan.


In [22]:
# ============================================================
# 4. LOAD DATASET
# ============================================================

test_predictions = pd.read_csv(TEST_PREDICTIONS_PATH)

print("Dataset berhasil dimuat.")
print("Shape test_predictions:", test_predictions.shape)

display(test_predictions.head())

Dataset berhasil dimuat.
Shape test_predictions: (4999, 16)


,timestamp,device_id,voltage,current,power,frequency,temperature,latency,packet_loss,throughput,duplicate_packet,checksum_valid,authentication_fail,DT_prediction,RF_prediction,LR_prediction
0,2024-01-01 11:34:30,SGD-0028,221.3982,3.9668,704.2116,49.7139,31.093,25.73,0.31,91.10,1,1,0,Normal,Normal,Normal
1,2024-01-01 03:44:00,SGD-0040,218.4783,4.8458,783.4720,50.2941,37.437,22.24,1.09,99.19,0,1,0,Normal,Normal,Normal
2,2024-01-01 12:10:35,SGD-0010,222.2007,7.5188,1903.1550,49.9098,38.242,11.59,0.75,95.27,0,1,0,Normal,Normal,Normal
3,2024-01-01 08:02:00,SGD-0040,225.0016,7.6916,1711.9549,50.1279,39.179,19.46,0.52,95.53,0,1,0,Normal,Normal,Normal
4,2024-01-01 12:30:55,SGD-0016,228.2348,3.1160,834.7887,49.8262,34.033,9.58,0.18,92.41,0,1,0,Normal,Normal,Normal


In [23]:
# ============================================================
# 5. CEK KOLOM DATASET
# ============================================================

print("Daftar kolom test_predictions:")
print(test_predictions.columns.tolist())

Daftar kolom test_predictions:
['timestamp', 'device_id', 'voltage', 'current', 'power', 'frequency', 'temperature', 'latency', 'packet_loss', 'throughput', 'duplicate_packet', 'checksum_valid', 'authentication_fail', 'DT_prediction', 'RF_prediction', 'LR_prediction']


In [6]:
# ============================================================
# 6. CEK KOLOM DATASET
# ============================================================

print("Kolom pada df_test:")
print(df_test.columns.tolist())

print("\nKolom pada test_predictions:")
print(test_predictions.columns.tolist())

Kolom pada df_test:
['timestamp', 'device_id', 'voltage', 'current', 'power', 'frequency', 'temperature', 'latency', 'packet_loss', 'throughput', 'duplicate_packet', 'checksum_valid', 'authentication_fail']

Kolom pada test_predictions:
['timestamp', 'device_id', 'voltage', 'current', 'power', 'frequency', 'temperature', 'latency', 'packet_loss', 'throughput', 'duplicate_packet', 'checksum_valid', 'authentication_fail', 'DT_prediction', 'RF_prediction', 'LR_prediction']


In [7]:
# ============================================================
# 7. VALIDASI JUMLAH BARIS
# ============================================================

if len(df_test) == len(test_predictions):
    print("Jumlah baris df_test dan test_predictions SAMA.")
    print("Jumlah baris:", len(df_test))
else:
    print("Jumlah baris TIDAK SAMA.")
    print("Jumlah baris df_test         :", len(df_test))
    print("Jumlah baris test_predictions:", len(test_predictions))

Jumlah baris df_test dan test_predictions SAMA.
Jumlah baris: 4999


In [8]:
# ============================================================
# 8. VALIDASI KOLOM LABEL ASLI
# ============================================================

target_column = "label"

if target_column in df_test.columns:
    print("Kolom label ditemukan di df_test.")
else:
    print("Kolom label tidak ditemukan di df_test.")
    print("Kolom yang tersedia:", df_test.columns.tolist())

Kolom label tidak ditemukan di df_test.
Kolom yang tersedia: ['timestamp', 'device_id', 'voltage', 'current', 'power', 'frequency', 'temperature', 'latency', 'packet_loss', 'throughput', 'duplicate_packet', 'checksum_valid', 'authentication_fail']


### Ambil DT, LR, RF 

In [13]:
# ============================================================
# 4. AMBIL KOLOM PREDIKSI MODEL INDIVIDUAL
# ============================================================

prediction_columns = [
    "DT_prediction",
    "RF_prediction",
    "LR_prediction"
]

# Ambil hanya kolom prediksi dari test_predictions
model_predictions = test_predictions[prediction_columns].copy()

print("Kolom prediksi model individual berhasil diambil.")
print("Shape model_predictions:", model_predictions.shape)

display(model_predictions.head())

Kolom prediksi model individual berhasil diambil.
Shape model_predictions: (4999, 3)


,DT_prediction,RF_prediction,LR_prediction
0,Normal,Normal,Normal
1,Normal,Normal,Normal
2,Normal,Normal,Normal
3,Normal,Normal,Normal
4,Normal,Normal,Normal


In [14]:
# ============================================================
# 4.1 VALIDASI NILAI PREDIKSI
# ============================================================

valid_labels = [0, 1, 2]

for col in prediction_columns:
    unique_values = sorted(model_predictions[col].unique())
    
    print(f"Nilai unik pada {col}:")
    print(unique_values)
    
    invalid_values = [
        value for value in unique_values
        if value not in valid_labels
    ]
    
    if len(invalid_values) == 0:
        print(f"{col} hanya berisi label valid.")
    else:
        print(f"{col} memiliki label tidak valid:")
        print(invalid_values)
    
    print("-" * 50)

Nilai unik pada DT_prediction:
['Attack', 'Fault', 'Normal']
DT_prediction memiliki label tidak valid:
['Attack', 'Fault', 'Normal']
--------------------------------------------------
Nilai unik pada RF_prediction:
['Attack', 'Fault', 'Normal']
RF_prediction memiliki label tidak valid:
['Attack', 'Fault', 'Normal']
--------------------------------------------------
Nilai unik pada LR_prediction:
['Attack', 'Fault', 'Normal']
LR_prediction memiliki label tidak valid:
['Attack', 'Fault', 'Normal']
--------------------------------------------------


In [15]:
# ============================================================
# 4.2 CEK MISSING VALUE PADA KOLOM PREDIKSI
# ============================================================

print("Missing value pada kolom prediksi:")

prediction_missing = model_predictions.isnull().sum()

print(prediction_missing)

if prediction_missing.sum() == 0:
    print("\nTidak ada missing value pada kolom prediksi.")
else:
    print("\nAda missing value pada kolom prediksi. Perlu ditangani dulu.")

Missing value pada kolom prediksi:
DT_prediction    0
RF_prediction    0
LR_prediction    0
dtype: int64

Tidak ada missing value pada kolom prediksi.


In [16]:
# ============================================================
# 4.3 CEK DISTRIBUSI PREDIKSI TIAP MODEL
# ============================================================

label_mapping = {
    0: "normal",
    1: "attack",
    2: "fault"
}

for col in prediction_columns:
    print(f"Distribusi prediksi {col}:")
    print(
        model_predictions[col]
        .value_counts()
        .sort_index()
        .rename(index=label_mapping)
    )
    print("-" * 50)

Distribusi prediksi DT_prediction:
DT_prediction
Attack    2149
Fault     1146
Normal    1704
Name: count, dtype: int64
--------------------------------------------------
Distribusi prediksi RF_prediction:
RF_prediction
Attack    2159
Fault     1155
Normal    1685
Name: count, dtype: int64
--------------------------------------------------
Distribusi prediksi LR_prediction:
LR_prediction
Attack    2130
Fault      977
Normal    1892
Name: count, dtype: int64
--------------------------------------------------


In [17]:
# ============================================================
# 4.3 CEK DISTRIBUSI PREDIKSI TIAP MODEL
# ============================================================

label_mapping = {
    0: "normal",
    1: "attack",
    2: "fault"
}

for col in prediction_columns:
    print(f"Distribusi prediksi {col}:")
    print(
        model_predictions[col]
        .value_counts()
        .sort_index()
        .rename(index=label_mapping)
    )
    print("-" * 50)

Distribusi prediksi DT_prediction:
DT_prediction
Attack    2149
Fault     1146
Normal    1704
Name: count, dtype: int64
--------------------------------------------------
Distribusi prediksi RF_prediction:
RF_prediction
Attack    2159
Fault     1155
Normal    1685
Name: count, dtype: int64
--------------------------------------------------
Distribusi prediksi LR_prediction:
LR_prediction
Attack    2130
Fault      977
Normal    1892
Name: count, dtype: int64
--------------------------------------------------


In [18]:
# ============================================================
# 4.4 GABUNGKAN PREDIKSI MODEL DENGAN LABEL ASLI
# ============================================================

target_column = "label"

voting_data = model_predictions.copy()

# Tambahkan label asli dari df_test
voting_data["actual_label"] = df_test[target_column].values

# Tambahkan nama kelas agar mudah dibaca
voting_data["actual_class"] = voting_data["actual_label"].map(label_mapping)

print("Data untuk proses voting berhasil dibuat.")
print("Shape voting_data:", voting_data.shape)

display(voting_data.head())

KeyError: 'label'